In [19]:
# =========================
# 1. INSTALLS / IMPORTS
# =========================
from __future__ import annotations

import os
import re
import json
import math
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dataclasses import dataclass, field, asdict
from collections import defaultdict

import pandas as pd
import numpy as np

from dotenv import load_dotenv

load_dotenv()

True

In [20]:
# =========================
# 2. CONFIG
# =========================

DATA_ROOT = Path("/Users/ananthakrishnab/Desktop/Projects/Hackthon-IIT/data/Claims")         # input claims folder
OUTPUT_ROOT = Path("./outputs")    # json/csv/html outputs
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

PACKAGE_CODES = ["MG064A", "SG039C", "MG006A", "SB039A"]

DECISION_PASS = "PASS"
DECISION_CONDITIONAL = "CONDITIONAL"
DECISION_FAIL = "FAIL"

CLIENT_ID = os.getenv("CLIENT_ID")
CLINENT_SECRET = os.getenv("CLINENT_SECRET")

In [21]:
# =========================
# 2.1 CONFIG (MODELS)
# =========================

import json
import logging
import httpx

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)


class NHAclient:
    TOKEN_URL = "https://aaehackathon.nhaad.in/controlplane/iudx/v2/auth/token"
    COMPLETIONS_URL = "https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions"

    def __init__(self, client_id: str, client_secret: str):
        self.id = client_id
        self.secret = client_secret
        self.token = self._get_token()

    def _get_token(self) -> str:
        logger.info("Fetching new auth token...")
        resp = httpx.post(
            self.TOKEN_URL,
            headers={
                "clientId": self.id,
                "clientSecret": self.secret
            },
            follow_redirects=True
        )
        resp.raise_for_status()
        data = resp.json()
        
        result = data.get("result", {})
        token = result.get("access_token") or result.get("token")
        if not token:
            raise ValueError(f"Token not found in response: {data}")
        
        logger.info("Auth token obtained successfully")
        return token

    def completion(self, model: str, messages: list, **kwargs) -> dict:
        payload = {
            "model": model,
            "messages": messages,
            "auth_token": self.token,
            **kwargs
        }

        with httpx.Client(timeout=None) as client:
            resp = client.post(self.COMPLETIONS_URL, json=payload)
            
            if self._is_token_expired(resp):
                logger.info("Token expired, refreshing token and retrying...")
                self.token = self._get_token()
                payload["auth_token"] = self.token
                resp = client.post(self.COMPLETIONS_URL, json=payload)
            
            return resp.json()

    def _is_token_expired(self, resp: httpx.Response) -> bool:
        if resp.status_code == 401:
            return True
        
        if resp.status_code == 200:
            try:
                data = resp.json()
                if isinstance(data, dict):
                    error = data.get("error", {})
                    if isinstance(error, dict):
                        message = error.get("message", "")
                        if "expired" in message.lower() or "token" in message.lower():
                            return True
            except Exception:
                pass
        
        return False

In [22]:
# Testing the Model Client
def main():
    client = NHAclient(
        client_id=CLIENT_ID,
        client_secret=CLINENT_SECRET
    )
    
    resp = client.completion(
        model="Gemma 3 4B",
        messages=[{"role": "user", "content": [{"type": "text", "text" : "Say hello in 3 words"}]}],
        metadata={
            "problem_statement":1
        }
    )
    print(resp)

main()

# response = nc.completion(
#     model="Gemma 3 4B", #use one of the models
#     messages=[
#         {
#             "role": "user",
#             "content": [
#                 {"type": "image_url", "image_url": {"url": data_url}},
#                 {"type": "text", "text": "What do you see"},
#             ],
#         }
#     ],
#     metadata={
#             "problem_statement":1

2026-04-22 15:24:59,619 - INFO - Fetching new auth token...
2026-04-22 15:24:59,841 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/controlplane/iudx/v2/auth/token "HTTP/1.1 200 OK"
2026-04-22 15:24:59,842 - INFO - Auth token obtained successfully
2026-04-22 15:25:00,100 - INFO - HTTP Request: POST https://aaehackathon.nhaad.in/nhm-proxy/v1/chat/completions "HTTP/1.1 200 OK"


{'id': 'chatcmpl-485561c8-4d9f-47b8-ab15-e07d3e5777b2', 'created': 1776851700, 'model': 'google.gemma-3-4b-it', 'object': 'chat.completion', 'system_fingerprint': None, 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': 'Hello there, friend!', 'role': 'assistant', 'tool_calls': None, 'function_call': None}}], 'usage': {'completion_tokens': 6, 'prompt_tokens': 15, 'total_tokens': 21, 'completion_tokens_details': {'reasoning_tokens': 0, 'text_tokens': 6}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': None, 'image_tokens': None, 'video_tokens': None, 'cache_creation_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0}}


In [23]:
# =========================
# 3. OUTPUT SCHEMAS
# =========================
# These are the exact expected keys per package, based on the provided examples.

PACKAGE_SCHEMAS = {
    "MG064A": [
        "case_id", "link", "procedure_code", "page_number",
        "clinical_notes", "cbc_hb_report", "indoor_case",
        "treatment_details", "post_hb_report", "discharge_summary",
        "severe_anemia", "common_signs", "significant_signs",
        "life_threatening_signs", "extra_document", "document_rank"
    ],
    "SG039C": [
        "case_id", "S3_link/DocumentName", "procedure_code", "page_number",
        "clinical_notes", "usg_report", "lft_report", "operative_notes",
        "pre_anesthesia", "discharge_summary", "photo_evidence",
        "histopathology", "clinical_condition", "usg_calculi",
        "pain_present", "previous_surgery", "extra_document", "document_rank"
    ],
    "MG006A": [
        "case_id", "S3_link/DocumentName", "procedure_code", "page_number",
        "clinical_notes", "investigation_pre", "pre_date", "vitals_treatment",
        "investigation_post", "post_date", "discharge_summary", "poor_quality",
        "fever", "symptoms", "extra_document", "document_rank"
    ],
    "SB039A": [
        "case_id", "link", "procedure_code", "page_number",
        "clinical_notes", "xray_ct_knee", "indoor_case", "operative_notes",
        "implant_invoice", "post_op_photo", "post_op_xray", "discharge_summary",
        "doa", "dod", "arthritis_type", "post_op_implant_present",
        "age_valid", "extra_document", "document_rank"
    ],
}

KEY_ALIASES = {
    "S3_link": "link",
    "s3_link": "link",
    "S3_link/DocumentName": "link",
}

In [24]:
# =========================
# 4. DATA MODELS
# =========================

@dataclass
class OCRLine:
    text: str
    bbox: Optional[List[int]] = None
    confidence: Optional[float] = None

@dataclass
class PageResult:
    case_id: str
    file_name: str
    page_number: int
    extracted_text: str = ""
    ocr_lines: List[OCRLine] = field(default_factory=list)
    doc_type: str = "unknown"
    doc_type_confidence: float = 0.0
    visual_tags: Dict[str, Any] = field(default_factory=dict)
    entities: Dict[str, Any] = field(default_factory=dict)
    quality: Dict[str, Any] = field(default_factory=dict)
    output_row: Dict[str, Any] = field(default_factory=dict)
    evidence: Dict[str, Any] = field(default_factory=dict)

@dataclass
class TimelineEvent:
    sequence: int
    event_type: str
    date: Optional[str]
    source_document: str
    temporal_validity: str
    evidence: Dict[str, Any] = field(default_factory=dict)

@dataclass
class ClaimDecision:
    case_id: str
    package_code: str
    decision: str
    confidence: float
    reasons: List[str]
    missing_documents: List[str] = field(default_factory=list)
    rule_flags: List[str] = field(default_factory=list)
    timeline_flags: List[str] = field(default_factory=list)

# 1. INGEST CLAIM FILES


In [25]:
# =========================
# 1. INGEST CLAIM FILES
# =========================

def iter_case_files(case_dir: Path) -> List[Path]:
    sorted_list = sorted([
        p for p in case_dir.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS])

    return sorted_list
    

def discover_cases(data_root: Path) -> Dict[str, List[Path]]:
    cases = {}
    for case_dir in sorted(data_root.iterdir()):
        if case_dir.is_dir():
            files = iter_case_files(case_dir)
            if files:
                cases[case_dir.name] = files
    return cases

In [33]:
cases = discover_cases(DATA_ROOT)
print(cases.keys())

dict_keys(['MG006A', 'MG064A', 'SB039A', 'SG039C'])


# 2. SPLIT PDFS/IMAGES INTO PAGES

In [34]:
# =========================
# 2. SPLIT PDFS/IMAGES INTO PAGES
# =========================

from pdf2image import convert_from_path
from PIL import Image

def extract_pages(file_path):
    pages = []
    suffix = file_path.suffix.lower()

    if suffix == ".pdf":
        try:
            images = convert_from_path(str(file_path), dpi=150)
            for page_num, img in enumerate(images, start=1):
                pages.append({
                    "page_number": page_num,
                    "image": img,
                    "file_name": file_path.name
                })
        except Exception as e:
            print(f"[WARN] Failed to process PDF {file_path.name}: {e}")
    else:
        try:
            img = Image.open(str(file_path))
            img.load()
            pages.append({"page_number": 1, "image": img, "file_name": file_path.name})
        except Exception as e:
            print(f"[WARN] Failed to open image {file_path.name}: {e}")

    return pages

In [37]:
CASE_ID = list(cases.keys())
all_pages = []

for f in cases[case_id]:
    all_pages.extend(extract_pages(f))

for p in all_pages:
    print(f"  [{p['page_number']}] {p['file_name']} → {p['image'].size}")

  [1] 000585__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDHAN_DB.pdf → (1241, 1754)
  [2] 000585__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDHAN_DB.pdf → (1241, 1754)
  [3] 000585__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDHAN_DB.pdf → (1241, 1754)
  [4] 000585__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDHAN_DB.pdf → (1241, 1754)
  [1] 000590__CMJAY_TR_CMJAY_2025_R3_1022010623__36acf382-6069-49c4-b705-a1c62a644a67.jpg → (1161, 1600)
  [1] 000591__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDAM.pdf → (1241, 1754)
  [2] 000591__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDAM.pdf → (1241, 1754)
  [3] 000591__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDAM.pdf → (1241, 1754)
  [4] 000591__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDAM.pdf → (1241, 1754)
  [5] 000591__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDAM.pdf → (1241, 1754)
  [6] 000591__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDAM.pdf → (1241, 1754)
  [7] 000591__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDAM.pdf → (1241, 1754)
  [8] 000591__CMJAY_TR_CMJAY_2025_R3_1022010623__SUDAM.pdf → (124

# 3. OCR EACH PAGE

In [17]:
# =========================
# 3. OCR EACH PAGE
# =========================

def run_ocr(page_image: Any) -> Tuple[str, List[OCRLine]]:
    '''
    Extract text and line-level OCR evidence from a single page image.

    Expected behavior when implemented:
    - run OCR on the provided page image
    - return the full extracted text as a single string
    - return structured OCR lines with optional bounding boxes/confidence
    - support mixed-quality and multilingual claim documents where possible
    '''
    pass
